# Day 29 — Practice Questions
### Correlation · Module 2 · Topic 2.9

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it tests | Count |
|---|---|---|
| **A** | Predict the output | 8 |
| **B** | Spot & fix the bug | 4 |
| **C** | Write the code | 5 |
| **D** | Challenge | 2 |

**How to work:** for Part A, write your answer down *before* running anything.

Three questions to keep asking yourself:

- **Did I plot it?**
- **How many pairs did I look at?**
- **Which of the five explanations is this?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

print("ready")

---
# Part A — Predict the output

Eight short cells. Write your prediction down first, then run it. The point is to find out
where your intuition about r is wrong, so a wrong guess is more useful than a right one.

### A1 · Does rescaling y change r?

`y` is multiplied by 100 and shifted by 500. Predict all three numbers.

In [ ]:
rng = np.random.default_rng(101)
x = rng.normal(0, 1, 50)
y = 2*x + rng.normal(0, 1, 50)

print(round(stats.pearsonr(x, y).statistic, 4))
print(round(stats.pearsonr(x, 100*y + 500).statistic, 4))
print(round(stats.pearsonr(x, -y).statistic, 4))

### A2 · A perfect, noiseless parabola

There is no randomness here at all. `y` is determined by `x` exactly. What is r?

In [ ]:
x = np.linspace(-5, 5, 101)
print(round(stats.pearsonr(x, x**2).statistic, 4))

x2 = np.linspace(0, 5, 101)
print(round(stats.pearsonr(x2, x2**2).statistic, 4))

### A3 · Which way round?

In [ ]:
rng = np.random.default_rng(103)
a = rng.normal(0, 1, 200)
b = 0.7*a + rng.normal(0, 0.7, 200)

print(stats.pearsonr(a, b).statistic == stats.pearsonr(b, a).statistic)

### A4 · A steep but perfectly monotonic curve

In [ ]:
x = np.linspace(1, 5, 60)
y = x**5                       # no noise at all

print("Pearson :", round(stats.pearsonr(x, y).statistic, 4))
print("Spearman:", round(stats.spearmanr(x, y).statistic, 4))

### A5 · Three correlations, three shares of variance

In [ ]:
for r in [0.4, 0.6, 0.8]:
    print(f"r = {r}  ->  r^2 = {r**2:.0%}")

### A6 · One point, added to a strong negative relationship

In [ ]:
rng = np.random.default_rng(66)
x = rng.normal(0, 1, 30)
y = -x + rng.normal(0, 0.4, 30)

print("30 points      :", round(stats.pearsonr(x, y).statistic, 4))
print("+ one at (10,10):", round(stats.pearsonr(np.append(x, 10), np.append(y, 10)).statistic, 4))

### A7 · Two edge cases

In [ ]:
x = np.array([1., 2, 3, 4, 5])
c = np.array([7., 7, 7, 7, 7])

print("x with itself  :", stats.pearsonr(x, x).statistic)
print("x with constant:", stats.pearsonr(x, c).statistic)

### A8 · What is Spearman, really?

In [ ]:
rng = np.random.default_rng(8)
a = rng.normal(0, 1, 40)
b = np.exp(a)

print(round(stats.pearsonr(stats.rankdata(a), stats.rankdata(b)).statistic, 6))
print(round(stats.spearmanr(a, b).statistic, 6))

---
# Part B — Spot & fix the bug

Each cell runs without error and produces a confident, wrong conclusion. Find it, say why,
and write the fix.

### B1 · "There is no relationship between dose and effect"

A pharmacologist runs this and concludes the drug does nothing.

In [ ]:
rng = np.random.default_rng(310)
dose   = rng.uniform(0, 10, 120)
effect = -(dose - 5)**2 + 25 + rng.normal(0, 2.5, 120)

res = stats.pearsonr(dose, effect)
print(f"r = {res.statistic:.4f}, p = {res.pvalue:.4f}")
print("Conclusion: no relationship between dose and effect.")

**What is wrong, and what is the real finding?**

### B2 · "We found a significant driver of churn"

An analyst correlates every pair of twelve metrics and reports the strongest.

In [ ]:
rng = np.random.default_rng(311)
df = pd.DataFrame(rng.normal(0, 1, (50, 12)),
                  columns=[f"m{i:02d}" for i in range(12)])
# (all twelve columns are independent noise)

best = None
for i in range(12):
    for j in range(i + 1, 12):
        res = stats.pearsonr(df.iloc[:, i], df.iloc[:, j])
        if best is None or res.pvalue < best[3]:
            best = (df.columns[i], df.columns[j], res.statistic, res.pvalue)

print(f"{best[0]} and {best[1]}: r = {best[2]:.3f}, p = {best[3]:.5f}")
print("Significant at p < 0.05 -> worth investigating.")

**What is wrong, and what is the correct threshold?**

### B3 · "Strong evidence from our pilot study"

Eight customers were surveyed. The consultant reports the result.

In [ ]:
rng = np.random.default_rng(537)
satisfaction = rng.normal(0, 1, 8)
spend        = rng.normal(0, 1, 8)     # genuinely unrelated

res = stats.pearsonr(satisfaction, spend)
print(f"r = {res.statistic:.3f}, p = {res.pvalue:.4f}")
print("Satisfaction strongly predicts spend (p < 0.05).")

**What is missing, and what should the report say?**

### B4 · "Coffee machines drive revenue"

In [ ]:
rng = np.random.default_rng(313)
n = 300
company_size = rng.uniform(1, 10, n)                       # in hundreds of staff
coffee_machines = 3*company_size + rng.normal(0, 2, n)
revenue         = 50*company_size + rng.normal(0, 30, n)

res = stats.pearsonr(coffee_machines, revenue)
print(f"r = {res.statistic:.3f}, p = {res.pvalue:.2e}")
print("Recommendation: install more coffee machines to increase revenue.")

**Which of the five explanations is this, and how do you demonstrate it?**

---
# Part C — Write the code

Five tasks. Write the function, then run the test cell underneath it.

### C1 · A correlation report you could paste into a document

Write `report(x, y, n_pairs=1)` that prints **all five numbers** a reader needs: n, r with
its 95% confidence interval, Spearman, r², and the slope. It should also warn when Pearson
and Spearman disagree by more than 0.15.

In [ ]:
def report(x, y, n_pairs=1):
    # your code here
    pass

### C2 · How much does any single point matter?

Write `most_influential(x, y)` that removes each point in turn, recomputes r, and reports
which single point moves r the most — and by how much.

In [ ]:
rng = np.random.default_rng(320)
x = np.append(rng.normal(0, 1, 60), 6.0)
y = np.append(0.1*rng.normal(0, 1, 60) + rng.normal(0, 1, 60), 6.0)
# note: x and y are essentially unrelated, except for that last point

def most_influential(x, y):
    # your code here
    pass

### C3 · Partial correlation

Shoe size and reading ability in schoolchildren. Write `partial_corr(a, b, control)` and
use it to work out what is really going on.

In [ ]:
rng = np.random.default_rng(321)
n = 400
age     = rng.uniform(5, 16, n)
shoe    = 18 + 0.9*age + rng.normal(0, 0.8, n)
reading = 10 + 4.5*age + rng.normal(0, 6, n)

print("shoe vs reading: r = %.4f" % stats.pearsonr(shoe, reading).statistic)

def partial_corr(a, b, control):
    # your code here
    pass

### C4 · Scan a table for Pearson/Spearman disagreements

Write `disagreements(df, gap=0.15)` that compares the two coefficients for every pair of
columns and returns only the pairs where they differ by more than `gap`. This is a
one-glance data-quality check you can run on any new dataset.

In [ ]:
rng = np.random.default_rng(322)
df = pd.DataFrame({"a": rng.normal(0, 1, 80),
                   "b": rng.normal(0, 1, 80),
                   "c": rng.normal(0, 1, 80)})
df["d"] = np.exp(2*df.a)          # a monotonic curve
df["e"] = df.b.copy()
df.loc[0, "e"] = 25               # one bad value

def disagreements(df, gap=0.15):
    # your code here
    pass

### C5 · How much data do you actually need?

Two questions a project plan should answer before the data is collected:

1. What is the smallest n at which a true correlation of r would reach p < 0.05?
2. How many rows do you need for the 95% interval around r = 0.4 to be narrower than 0.20?

In [ ]:
def n_for_significance(r, alpha=0.05):
    # your code here
    pass

def ci_width(r, n):
    # your code here
    pass

---
# Part D — Challenge

Two open tasks. Both are short to code and long to think about.

### D1 · Manufacture a finding, then take it apart

You have been handed 20 business KPIs measured over 60 weeks. There is **nothing** in the
data — every column is independent noise.

1. Find the most "significant" pair.
2. Write the two-sentence summary a careless analyst would put in a deck.
3. Write the honest version.
4. State exactly what extra information turns version 2 into version 3.

In [ ]:
rng = np.random.default_rng(330)
cols = [f"kpi_{i:02d}" for i in range(20)]
df = pd.DataFrame(rng.normal(0, 1, (60, 20)), columns=cols)

# your code here

### D2 · Diagnose the explanation

A recruiting team reports: *"Among our hires, interview preparation is strongly **negatively**
correlated with years of experience (r = −0.65, p < 1e-42). Experienced candidates don't
bother preparing."*

The data below is the full applicant pool, with a `hired` flag.

1. Compute the correlation among the hired, and among everyone.
2. Decide which of the five explanations this is.
3. Say what the recruiting team should conclude instead.

In [ ]:
rng = np.random.default_rng(331)
N = 4000
prep       = rng.normal(0, 1, N)                          # interview preparation
experience = rng.normal(0, 1, N)                          # years of experience
hired      = (prep + experience + rng.normal(0, 0.3, N)) > 1.9

applicants = pd.DataFrame({"prep": prep, "experience": experience, "hired": hired})
print(applicants.hired.value_counts().to_dict())

# your code here

---
### Before you hand in

- For every correlation you computed, did you **plot it first**?
- Did you report **r², the confidence interval and n**, not just r?
- Did you check **Spearman** alongside Pearson, and explain any gap?
- Did you count **how many pairs** you examined, and say so?
- For every causal-sounding claim, did you name which of the **five explanations** it is?
- For Part D, did you write both the misleading version and the honest one? Both are marked.

Solutions will be released after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*